# MethodThinker Colab 训练

一键运行训练脚本 - 直接复制整个notebook到Colab执行

In [ ]:
# 1. 检查GPU
!nvidia-smi
print("\n✓ GPU检查完成")

In [ ]:
# 2. 克隆项目
!git clone https://github.com/alasong/method-thinker.git
%cd method-thinker
print("\n✓ 项目克隆完成")

In [ ]:
# 3. 安装依赖
!pip install -q transformers accelerate peft datasets bitsandbytes trl pyyaml
print("\n✓ 依赖安装完成")

In [ ]:
# 4. 挂载Google Drive
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/method-thinker-models
print("\n✓ Drive挂载完成")

In [ ]:
# 5. 快速训练 (约15分钟)
# 使用QLoRA 4-bit量化适合T4显存

!python scripts/train_sft.py \
    --base-model Qwen/Qwen2.5-Math-1.5B \
    --output-dir /content/drive/MyDrive/method-thinker-models/v1 \
    --mode methodology-injection \
    --use-lora \
    --lora-r 16 \
    --epochs 1 \
    --batch-size 1 \
    --gradient-accumulation 16 \
    --max-length 2048 \
    --learning-rate 5e-5 \
    --dry-run

# 注意：先运行dry-run验证配置，确认无误后去掉--dry-run正式训练

In [ ]:
# 6. 正式训练 (去掉--dry-run)
!python scripts/train_sft.py \
    --base-model Qwen/Qwen2.5-Math-1.5B \
    --output-dir /content/drive/MyDrive/method-thinker-models/v1 \
    --mode methodology-injection \
    --use-lora \
    --lora-r 16 \
    --epochs 3 \
    --batch-size 4 \
    --gradient-accumulation 8 \
    --max-length 2048 \
    --learning-rate 5e-5

In [ ]:
# 7. 验证模型
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_path = "/content/drive/MyDrive/method-thinker-models/v1"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.float16,
    device_map="auto"
)

# 测试推理
prompt = "解方程: x^2 - 5x + 6 = 0"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_new_tokens=256)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))